# Ćwiczenie 1 — IPython (Add-on): 01.06 Errors and Debugging

To jest **brakujący blok dydaktyczny** domykający wykład 1.  
Cel nie polega na „czytaniu teorii o błędach”, tylko na przejściu pełnego cyklu:

1. zobaczyć wyjątek,
2. przeczytać traceback,
3. użyć `%xmode`,
4. wejść do `%debug`,
5. a na końcu napisać **lepszą funkcję**, która zgłasza sensowny błąd.

Ten notebook najlepiej robić **po** głównym ćwiczeniu 1, a **przed** add-onem z `%prun` i `%lprun`.


## Jak pracować

- W miejscach `TODO` uzupełnij kod albo krótką odpowiedź.
- Zadania manualne (`%xmode`, `%debug`, `%pdb`) kończą się wpisaniem krótkiego wniosku do zmiennej.
- Po każdej sekcji uruchom testy z `assert`.
- Tu celowo są **prawdziwe błędy** — nie próbujemy ich od razu ukrywać.


In [2]:
from __future__ import annotations

import math
import traceback
import pandas as pd
import numpy as np
from IPython import get_ipython

ip = get_ipython()
assert ip is not None, "Ten notebook wymaga IPython/Jupyter."

def assert_nonempty_str(x: str, min_len: int = 40):
    assert isinstance(x, str), f"Expected str, got {type(x)}"
    assert len(x.strip()) >= min_len, f"Text too short (len={len(x.strip())}, min={min_len})"

toy = pd.DataFrame({
    "speed": [12.0, 15.0, 14.0, 11.0],
    "temp_c": [5.0, 7.0, 6.0, 8.0],
    "city": ["waw", "krk", "waw", "poz"],
})

print("Setup OK ✅")
toy


Setup OK ✅


,speed,temp_c,city
0,12.0,5.0,waw
1,15.0,7.0,krk
2,14.0,6.0,waw
3,11.0,8.0,poz


## 01.06 — Errors and Debugging


### Ćwiczenie 6.1 — Rozpoznaj wyjątek i przeczytaj traceback

Poniżej jest łańcuch wywołań z celowym błędem.  
Twoje zadanie:

1. uruchom `run_bug_case()`,
2. sprawdź **jaki to wyjątek**,
3. wpisz do zmiennych:
   - typ wyjątku,
   - nazwę błędnego klucza,
   - krótki komentarz: **gdzie** w łańcuchu wywołań pojawia się prawdziwy problem.


In [ ]:
def normalize_column(df: pd.DataFrame, col: str) -> pd.Series:
    s = df[col]
    return (s - s.mean()) / s.std()

def compute_feature(df: pd.DataFrame, col_name: str) -> pd.Series:
    key = col_name.strip().lower()
    return normalize_column(df, key)

def run_bug_case() -> pd.Series:
    # Celowy błąd: w DataFrame mamy 'temp_c', a nie 'temp'
    return compute_feature(toy, " Temp ")

try:
    run_bug_case()
except Exception as e:
    print("type:", type(e).__name__)
    print("message:", e)


In [ ]:
EXC_TYPE = "KeyError"
BROKEN_LOOKUP = "temp"
TRACEBACK_NOTE = (
    "Prawdziwy problem pojawia się w funkcji `normalize_column`, gdy Pandas próbuje "
    "odczytać kolumnę `df['temp']`, której w ramce danych nie ma. "
    "Wyżej w łańcuchu wywołań `compute_feature()` tylko przygotowuje klucz, a `run_bug_case()` "
    "uruchamia cały scenariusz. Traceback jest ważny, bo pokazuje zarówno miejsce zgłoszenia "
    "wyjątku, jak i drogę, którą program do niego doszedł."
)

assert EXC_TYPE.strip() == "KeyError"
assert BROKEN_LOOKUP.strip() == "temp"
assert_nonempty_str(TRACEBACK_NOTE, min_len=100)
print("Ćwiczenie 6.1 ✅")


### Ćwiczenie 6.2 — `%xmode`: Plain vs Verbose

Wykonaj ręcznie:

```python
%xmode Plain
run_bug_case()
%xmode Verbose
run_bug_case()
```

Następnie wpisz krótko:
- co pokazuje `Verbose`, czego zwykle nie widać tak dobrze w `Plain`,
- kiedy na zajęciach / przy samodzielnej pracy warto przełączyć się na `Verbose`.


In [1]:
XMODE_PLAIN_DONE = True
XMODE_VERBOSE_DONE = True
ANSWER_XMODE = (
    "Tryb `Plain` daje krótki, czytelny traceback i dobrze nadaje się do szybkiego "
    "zorientowania się, gdzie program się wywrócił. "
    "Tryb `Verbose` pokazuje więcej kontekstu: pełniejsze ramki stosu, nazwy zmiennych "
    "i więcej informacji o miejscu błędu. "
    "W prostych przypadkach `Plain` bywa wygodniejszy, ale gdy błąd przechodzi przez kilka "
    "funkcji, `Verbose` ułatwia dojście do realnej przyczyny, a nie tylko do ostatniej linijki."
)

assert XMODE_PLAIN_DONE is True
assert XMODE_VERBOSE_DONE is True
assert_nonempty_str(ANSWER_XMODE, min_len=140)
print("Ćwiczenie 6.2 ✅")


NameError: name 'assert_nonempty_str' is not defined

### Ćwiczenie 6.3 — `%debug` po wyjątku

Po uruchomieniu `run_bug_case()` wykonaj:

```python
%debug
```

W debuggerze sprawdź przynajmniej:
- wartość zmiennej `key`,
- jakie kolumny ma `df`,
- w której funkcji naprawdę jesteś.

Następnie uzupełnij poniższe pola.


In [ ]:
DEBUG_KEY = "temp"
DEBUG_COLUMNS_TEXT = "speed,temp_c,city"
DEBUG_FUNCTION_NOTE = (
    "`%debug` pozwolił wejść do konkretnej ramki stosu i sprawdzić stan programu w momencie "
    "awarii: wartość `key`, zawartość `df.columns` oraz to, w której funkcji naprawdę jesteśmy. "
    "Sam traceback mówi, że był `KeyError`, ale nie daje tak wygodnej możliwości zadawania pytań "
    "programowi „od środka”. Dzięki debuggerowi od razu widać, że problemem nie jest obliczanie "
    "z-score, tylko niedopasowanie nazwy kolumny `temp` do faktycznie dostępnej `temp_c`."
)

assert DEBUG_KEY.strip() == "temp"
assert "temp_c" in DEBUG_COLUMNS_TEXT
assert_nonempty_str(DEBUG_FUNCTION_NOTE, min_len=120)
print("Ćwiczenie 6.3 ✅")


### Ćwiczenie 6.4 — `%pdb on` i auto-debugging

Włącz automatyczne wchodzenie do debuggera po wyjątku:

```python
%pdb on
run_bug_case()
```

Potem możesz wyłączyć:
```python
%pdb off
```

Nie sprawdzamy tutaj „wyniku w debuggerze”, tylko czy rozumiesz **po co** to narzędzie istnieje.


In [ ]:
AUTO_PDB_USED = True
ANSWER_PDB = (
    "`%pdb on` jest wygodne, gdy często poprawiamy podobne błędy i chcemy automatycznie wpadać "
    "do debuggera bez ręcznego wpisywania `%debug` po każdym wyjątku. "
    "Może jednak przeszkadzać przy krótkich, celowo psujących się eksperymentach, bo każde "
    "potknięcie zatrzymuje wykonanie i wybija z rytmu pracy."
)

assert AUTO_PDB_USED is True
assert_nonempty_str(ANSWER_PDB, min_len=100)
print("Ćwiczenie 6.4 ✅")


### Ćwiczenie 6.5 — Zamień „słaby błąd” w „dobry błąd”

Teraz zrób ważny krok dydaktyczny:  
zamiast zostawiać enigmatyczny `KeyError`, napisz funkcję, która zgłasza **czytelny** `ValueError`
z informacją:
- jakiej kolumny szukaliśmy,
- jakie kolumny są dostępne.

Dodatkowo:
- jeśli kolumna istnieje, ale ma odchylenie standardowe bliskie zero, też rzuć `ValueError`,
- w przeciwnym razie zwróć z-score tej kolumny.

To ćwiczenie jest ważne, bo łączy **debugowanie** z **pisaniem lepszego kodu**.


In [3]:
def safe_compute_feature(df: pd.DataFrame, col_name: str) -> pd.Series:
    # 1) normalizacja wejścia
    key = str(col_name).strip().lower()

    # 2) mapa: nazwa znormalizowana -> oryginalna nazwa kolumny
    col_map = {str(col).strip().lower(): col for col in df.columns}

    # 3) czytelny błąd dla brakującej kolumny
    if key not in col_map:
        available = ", ".join(map(str, df.columns))
        raise ValueError(
            f"Nie znaleziono kolumny '{key}'. Dostępne kolumny: {available}."
        )

    original_col = col_map[key]
    s = df[original_col]

    # 4) kontrola odchylenia standardowego
    std = float(s.std())
    if (not np.isfinite(std)) or abs(std) < 1e-12:
        raise ValueError(
            f"Kolumna '{original_col}' ma odchylenie standardowe bliskie zeru; "
            "nie da się policzyć sensownego z-score."
        )

    # 5) z-score
    return ((s - s.mean()) / std).rename(original_col)

# tests
good = safe_compute_feature(toy, " temp_c ")
assert isinstance(good, pd.Series)
assert len(good) == len(toy)
assert np.isfinite(good).all()

failed_missing = False
try:
    safe_compute_feature(toy, "temp")
except ValueError as e:
    failed_missing = True
    msg = str(e).lower()
    assert "temp" in msg
    assert "temp_c" in msg
assert failed_missing, "Brak czytelnego ValueError dla brakującej kolumny."

toy_constant = toy.copy()
toy_constant["constant"] = 1.0

failed_constant = False
try:
    safe_compute_feature(toy_constant, "constant")
except ValueError as e:
    failed_constant = True
    assert "constant" in str(e).lower()
assert failed_constant, "Brak ValueError dla kolumny stałej."

print("Ćwiczenie 6.5 ✅")


Ćwiczenie 6.5 ✅


### Mini-ściąga do `%debug`

Najczęściej przydają się:
- `p nazwa` — pokaż wartość zmiennej,
- `l` — pokaż fragment kodu,
- `u` / `d` — idź wyżej / niżej po stosie,
- `n` — następna linia,
- `c` — kontynuuj wykonanie,
- `q` — wyjdź z debuggera.

Na tych zajęciach nie chodzi o zapamiętanie wszystkich komend, tylko o nawyk:
**zobacz traceback → wejdź do `%debug` → sprawdź zmienne → popraw kod lub komunikat błędu.**


## Do oddania

1. Uzupełnione zmienne z krótkimi odpowiedziami.
2. Działająca funkcja `safe_compute_feature`.
3. Krótka refleksja (2-4 zdania):  
   co było bardziej pomocne w tym przykładzie — sam traceback, `%xmode Verbose` czy `%debug` — i dlaczego.
